In [1]:
## This makes crystal part of chain Z

import os

# Define the directory containing the PDB files
directory = '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d5_tube_OH/outputs'

# Function to relabel chains of HETATM entries to chain Z
def relabel_chains_in_pdb(file_path):
    # Read the contents of the PDB file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Create a list to store the modified lines
    modified_lines = []

    # Process each line in the PDB file
    for line in lines:
        if line.startswith('HETATM'):
            # Replace the chain identifier (position 22 in the PDB format) with 'Z'
            line = line[:21] + 'Z' + line[22:]
        modified_lines.append(line)

    # Write the modified contents back to the file
    with open(file_path, 'w') as file:
        file.writelines(modified_lines)

# Process each PDB file in the directory
for filename in os.listdir(directory):
    if filename.endswith('.pdb'):
        file_path = os.path.join(directory, filename)
        relabel_chains_in_pdb(file_path)
        print(f"Processed {filename}")

print("All files have been processed.")

Processed D10_HAp_rep4_hexamer_bb_010_d3_long_tube_OH.pdb
Processed d10_crystal.pdb
Processed D9_HAp_rep5_fiber_hexamer_bb_010_d3_long_tube_OH.pdb
Processed D9_HAp_rep4_hexamer_bb_010_d3_long_tube_OH.pdb
Processed b10_crystal.pdb
Processed d9_crystal.pdb
Processed D10_HAp_rep5_fiber_hexamer_bb_010_d3_long_tube_OH.pdb
All files have been processed.


In [2]:
#fix Asp and Glu
import os
from Bio import PDB

def find_non_glutamates_and_non_threonines_in_pdb(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    non_glutamates_and_non_threonines_positions = []

    for model in structure:
        for chain in model:
            for residue in chain:
                if residue.get_resname() not in ['GLU', 'ASP']:
                    non_glutamates_and_non_threonines_positions.append(f"{chain.id}{residue.id[1]}")

    return ' '.join(non_glutamates_and_non_threonines_positions)

def main():
    pdb_dir = '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d3_tube_OH/input'  # Adjust this path as necessary
    
    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return
    
    # Base directory for output tasks
    base_directory = pdb_dir

    # Create an outputs/ directory within the base directory
    outputs_dir = os.path.join(base_directory, "glutamates_fixed_outputs")
    os.makedirs(outputs_dir, exist_ok=True)

    # Define the base command
    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--model_type 'ligand_mpnn' "
        "--temperature 0.3 " #change if needed
        "--homo_oligomer 1 "
        "--pdb_path {file_path} "
        "--redesigned_residues '{non_glutamates_and_non_threonines}' "
        "--out_folder {outputs_dir} "
        "--batch_size 5 " #change if needed
        "--number_of_batches 5 " #changed if needed
        "--bias_AA 'R:-0.9' " #this makes Arg to have lower possibility to occur, change if needed
        "--pack_side_chains 1 "
        "--repack_everything 1"
    )
    
    glutamates_fixed_tasks_file = os.path.join(base_directory, "glutamates_fixed_tasks")

    with open(glutamates_fixed_tasks_file, "a") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                non_glutamates_and_non_threonines = find_non_glutamates_and_non_threonines_in_pdb(file_path)
                command = base_command.format(
                    file_path=file_path, 
                    non_glutamates_and_non_threonines=non_glutamates_and_non_threonines,
                    outputs_dir=outputs_dir
                )
                file.write(f"{command}\n")

    print(f"Glutamates fixed tasks file '{glutamates_fixed_tasks_file}' has been created.")

    # Define the SBATCH header, change the email to your email
    sbatch_header = """#!/bin/bash
#SBATCH -J mpnn
#SBATCH -p cpu
#SBATCH --cpus-per-task=1
#SBATCH -t 04:00:00
#SBATCH --mem=8g
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=your_email@example.com

# Get the task command from the glutamates_fixed_tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" glutamates_fixed_tasks)

# Execute the task command
eval $task
"""

    # Write the SBATCH script to a file
    sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
    with open(sbatch_file, "w") as file:
        file.write(sbatch_header)

    print(f"SBATCH script '{sbatch_file}' has been created.")

    # Command to submit the job array
    submit_command = "sbatch -a 1-$(cat glutamates_fixed_tasks | wc -l) digs_array_job.sh"
    print(f"To submit the job array, use the following command:\ncd {os.path.abspath(base_directory)}\n{submit_command}")

if __name__ == "__main__":
    main()

Glutamates fixed tasks file '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d3_tube_OH/input/glutamates_fixed_tasks' has been created.
SBATCH script '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d3_tube_OH/input/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d3_tube_OH/input
sbatch -a 1-$(cat glutamates_fixed_tasks | wc -l) digs_array_job.sh


In [1]:
#Free ligandmpnn without residues being fixed
import os
from Bio import PDB

def main():
    pdb_dir = '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d5_tube_OH/outputs'  # Adjust this path as necessary
    
    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return
    
    # Base directory for output tasks
    base_directory = pdb_dir

    # Create an outputs/ directory within the base directory
    outputs_dir = os.path.join(base_directory, "unfixed_outputs")
    os.makedirs(outputs_dir, exist_ok=True)

    # Define the base command
    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--model_type 'ligand_mpnn' "
        "--temperature 0.3 "
        "--homo_oligomer 1 "
        "--pdb_path {file_path} "
        "--out_folder {outputs_dir} "
        "--batch_size 4 "
        "--number_of_batches 1 "
        "--bias_AA 'R:-0.9' "
        "--pack_side_chains 1 "
        "--repack_everything 1"
    )
    
    unfixed_tasks_file = os.path.join(base_directory, "unfixed_tasks")

    with open(unfixed_tasks_file, "a") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                command = base_command.format(
                    file_path=file_path, 
                    outputs_dir=outputs_dir
                )
                file.write(f"{command}\n")

    print(f"Unfixed tasks file '{unfixed_tasks_file}' has been created.")

    # Define the SBATCH header, change the email to your email
    sbatch_header = """#!/bin/bash
#SBATCH -J mpnn
#SBATCH -p cpu
#SBATCH --cpus-per-task=1
#SBATCH -t 04:00:00
#SBATCH --mem=8g
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=your_email@example.com

# Get the task command from the unfixed_tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" unfixed_tasks)

# Execute the task command
eval $task
"""

    # Write the SBATCH script to a file
    sbatch_file = os.path.join(base_directory, "unfixed_digs_array_job.sh")
    with open(sbatch_file, "w") as file:
        file.write(sbatch_header)

    print(f"SBATCH script '{sbatch_file}' has been created.")

    # Command to submit the job array
    submit_command = "sbatch -a 1-$(cat unfixed_tasks | wc -l) unfixed_digs_array_job.sh"
    print(f"To submit the job array, use the following command:\ncd {os.path.abspath(base_directory)}\n{submit_command}")

if __name__ == "__main__":
    main()

Unfixed tasks file '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d5_tube_OH/outputs/unfixed_tasks' has been created.
SBATCH script '/home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d5_tube_OH/outputs/unfixed_digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /home/tracyy2/git/tip_atom_mineralization/chr_diffusion_multichain/C3/4_ligandmpnn/010_d5_tube_OH/outputs
sbatch -a 1-$(cat unfixed_tasks | wc -l) unfixed_digs_array_job.sh
